# PTCG 915+ Lucario v4: Evaluation Log + Dual-Slot Strategy

This notebook keeps the reproducible public baseline workflow: **Run All** writes `main.py`, writes `deck.csv`, copies the official `cg/` engine, applies the v3 anti-Crustle patch, and builds `submission.tar.gz`.

The v3 update is a conservative upgrade on top of the 915+ Lucario search baseline. After reviewing public notebooks and the early meta, the highest-signal changes were:

- `penguin069/public-scores-915`: strong Mega Lucario ex + optional forward-search baseline.
- `romanrozen/strong-start-crustle-lucario-agent-v6/v7` and `kacchan/crustle-aware-mega-lucario-ex-anti-wall`: Crustle wall counterplay through non-ex Hariyama routing.
- `kojimar/validated-rule-based-agent-matchup-tests`: validation structure and archive checks.
- `pilkwang/pokemon-tcg-lucario-v2-strategy-baseline`: notebook EDA and diagnostics for action-space / fallback behavior.

What changed in v3:

- Keeps the proven Mega Lucario ex deck and search hook.
- Adds **Crustle-aware planning** so Mega Lucario ex does not repeatedly swing for 0 into Crustle (`345`).
- Routes energy/search priority toward **Makuhita -> Hariyama**, the in-deck non-ex answer.
- Fixes optional action selection so negative-score choices are not taken just because `maxCount` allows them.
- Adds an anti-stall `END` guard for very long turns.

**What changed in v4 (this version)**

- Added a public **evaluation log** after the first settled scores from the June 22 dual-slot experiment.
- Documents why we stopped chasing Dragapult as a scored slot and paired **Nithin 1084.5** with **Roman V10**.
- Explains the rule that only the **latest two submissions** count, so every test must be a deliberate pair.

Local smoke tests on the generated agent passed: mirror/random games and a Crustle sparring check completed with 0 engine errors. This is still a public research notebook, not a promise of first place; the active scored slots now follow the best reproduced public packages documented below.


In [ ]:
from pathlib import Path
from collections import Counter

DECK = [673, 673, 674, 674, 675, 675, 676, 676, 676, 677, 677, 677, 678, 678, 678, 678, 1102, 1102, 1102, 1102, 1123, 1123, 1141, 1141, 1141, 1141, 1142, 1142, 1142, 1142, 1152, 1152, 1152, 1152, 1159, 1182, 1182, 1192, 1192, 1192, 1192, 1227, 1227, 1227, 1227, 1252, 1252, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6]
assert len(DECK) == 60
counts = Counter(DECK)
assert all(n <= 4 for cid, n in counts.items() if cid != 6)
assert counts.get(1159, 0) <= 1
Path('deck.csv').write_text(''.join(f'{cid}\n' for cid in DECK))
print('wrote deck.csv', len(DECK), 'cards', len(counts), 'unique ids')


## Agent Structure

The agent has three layers. First, it scores legal options with a Lucario-specific heuristic. Second, if the deployed `cg.api` exposes the forward-search API, it evaluates a small candidate set under a per-decision time budget. Third, if anything unexpected happens, it falls back to a legal `minCount`/`maxCount` selection instead of throwing.

The practical reason to keep the wrapper is simple: in this competition, an exception is often worse than a merely mediocre move.


In [ ]:
%%writefile main.py
from __future__ import annotations
import os, sys, time, random
from collections import defaultdict

from cg.api import (
    AreaType, Card, CardType, EnergyType, Observation, OptionType,
    Pokemon, SelectContext, all_card_data, to_observation_class,
)

# ---------- Forward Search API (optional) ----------
_SEARCH_OK = False
try:
    from cg.api import search_begin, search_step, search_end, SearchState
    _SEARCH_OK = True
except Exception:
    pass

# ============================================================
# CONFIGURATION
# ============================================================
USE_SEARCH      = True
SEARCH_BUDGET   = 1.8        # seconds per decision
SEARCH_CANDS    = 6          # top-k actions to evaluate
SEARCH_DEPTH    = 40         # max forward steps
LOW_DECK        = 8          # anti-deckout threshold
MEGA_BRAVE_ID   = 983        # attack ID for Mega Brave

# ============================================================
# CARD IDS  (Mega Lucario ex Deck)
# ============================================================
class C:
    MAKUHITA             = 673
    HARIYAMA             = 674
    LUNATONE             = 675
    SOLROCK              = 676
    RIOLU                = 677
    MEGA_LUCARIO_EX      = 678
    BASIC_FIGHTING       = 6
    DUSK_BALL            = 1102
    SWITCH               = 1123
    PREMIUM_POWER_PRO    = 1141
    FIGHTING_GONG        = 1142
    POKE_PAD             = 1152
    HERO_CAPE            = 1159
    BOSS_ORDERS          = 1182
    CARMINE              = 1192
    LILLIE_DETERMINATION = 1227
    GRAVITY_MOUNTAIN     = 1252
    LUMIOSE_CITY         = 1267
    LILLIES_PEARL        = 1172
    LEGACY_ENERGY        = 12

# ---------- Hardcoded deck (60 cards) ----------
MY_DECK = [
    673,673, 674,674, 675,675, 676,676,676,
    677,677,677, 678,678,678,678,
    1102,1102,1102,1102, 1123,1123,
    1141,1141,1141,1141, 1142,1142,1142,1142,
    1152,1152,1152,1152, 1159,
    1182,1182, 1192,1192,1192,1192,
    1227,1227,1227,1227, 1252,1252,
    6,6,6,6,6,6,6,6,6,6,6,6,6,
]

# ---------- Card metadata ----------
_all = all_card_data()
CARD_DB = {c.cardId: c for c in _all}

# ---------- Global state ----------
_plan    = None
_turn    = -1
_ab_used = False

# ============================================================
# HELPERS
# ============================================================
def _get(obs, area, idx, pi):
    try:
        ps = obs.current.players[pi]
        match area:
            case AreaType.DECK:    return obs.select.deck[idx]
            case AreaType.HAND:    return ps.hand[idx]
            case AreaType.DISCARD: return ps.discard[idx]
            case AreaType.ACTIVE:  return ps.active[idx]
            case AreaType.BENCH:   return ps.bench[idx]
            case AreaType.PRIZE:   return ps.prize[idx]
            case AreaType.STADIUM: return obs.current.stadium[idx]
            case AreaType.LOOKING: return obs.current.looking[idx]
    except Exception:
        pass
    return None

def _prizes(p):
    d = CARD_DB.get(p.id)
    if d is None: return 1
    n = 3 if d.megaEx else 2 if d.ex else 1
    for c in p.energyCards:
        if c.id == C.LEGACY_ENERGY: n -= 1
    for c in p.tools:
        if c.id == C.LILLIES_PEARL and d.name and "Lillie" in d.name: n -= 1
    return max(0, n)

def _tgt_score(p):
    d = CARD_DB.get(p.id)
    s = _prizes(p) * 1000 + len(p.energies) * 150 + len(p.tools) * 100
    if d:
        s += 250 if d.stage2 else 130 if d.stage1 else 0
    if p.id in (144, 322, 323, 337): s -= 200
    if p.id == 112 and len(p.energies) >= 1: s += 300
    s += p.hp
    return s

# ============================================================
# ATTACK PLAN
# ============================================================
class Plan:
    __slots__ = ('atk','tgt','aidx','rhp','need_e')
    def __init__(self, atk=-1, tgt=-1, aidx=-1, rhp=-1, ne=False):
        self.atk=atk; self.tgt=tgt; self.aidx=aidx; self.rhp=rhp; self.need_e=ne

# ============================================================
# POLICY ENGINE
# ============================================================
class Policy:
    def __init__(self, obs):
        self.obs = obs
        self.st  = obs.current
        self.sel = obs.select
        self.ctx = self.sel.context
        self.mi  = self.st.yourIndex
        self.me  = self.st.players[self.mi]
        self.op  = self.st.players[1 - self.mi]
        self.mp  = len(self.me.prize)
        self.fc  = defaultdict(int)
        self.hc  = defaultdict(int)
        self.dc  = defaultdict(int)
        self.rdy_luc = False
        self.rdy_har = False
        self.sw = False; self.gust = False
        self.atk = False; self.mb = False
        self.sid = self.st.stadium[0].id if self.st.stadium else 0
        self._count(); self._scan()

    def choose(self):
        global _plan
        if self.ctx == SelectContext.MAIN:
            _plan = self._make_plan()
        scores = [self._sc(o) for o in self.sel.option]
        ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        self._track_ab(ranked)
        return ranked

    # ---------- counting ----------
    def _count(self):
        for p in self.me.active + self.me.bench:
            if p is None: continue
            self.fc[p.id] += 1
            if p.id in (C.MAKUHITA, C.HARIYAMA) and len(p.energies) >= 3: self.rdy_har = True
            if p.id in (C.RIOLU, C.MEGA_LUCARIO_EX) and len(p.energies) >= 2: self.rdy_luc = True
        for c in self.me.hand:    self.hc[c.id] += 1
        for c in self.me.discard: self.dc[c.id] += 1

    def _scan(self):
        if self.ctx != SelectContext.MAIN: return
        for o in self.sel.option:
            if o.type == OptionType.PLAY:
                c = _get(self.obs, AreaType.HAND, o.index, self.mi)
                if c and c.id == C.SWITCH: self.sw = True
                if c and c.id == C.BOSS_ORDERS: self.gust = True
            elif o.type == OptionType.EVOLVE:
                c = _get(self.obs, AreaType.HAND, o.index, self.mi)
                if c and c.id == C.HARIYAMA: self.gust = True
            elif o.type == OptionType.RETREAT: self.sw = True
            elif o.type == OptionType.ATTACK:
                self.atk = True
                if o.attackId == MEGA_BRAVE_ID: self.mb = True

    # ---------- attack plan ----------
    def _base_atk(self, p, ai, bi):
        er = bd = bs = 0
        if p.id == C.MEGA_LUCARIO_EX:
            if ai == 0: er, bd = 1, 130; bs += 60 * min(3, self.dc[C.BASIC_FIGHTING])
            else:       er, bd = 2, 270
            if self.mp in (2, 3): bs -= 500
        elif ai == 1: return None
        elif p.id == C.HARIYAMA: er, bd = 3, 210
        elif p.id == C.MAKUHITA:
            if not self._can_evo(bi): return None
            er, bd, bs = 3, 210, -100
        elif p.id == C.SOLROCK and self.fc[C.LUNATONE] >= 1: er, bd = 1, 70
        return (er, bd, bs) if bd > 0 else None

    def _can_evo(self, bi):
        for o in self.sel.option:
            if o.type != OptionType.EVOLVE: continue
            ti = o.inPlayIndex + (1 if o.inPlayArea == AreaType.BENCH else 0)
            if ti == bi: return True
        return False

    def _make_plan(self):
        best_s, best_p = -1, Plan()
        if self.st.turn < 2: return best_p
        board_me = [self.me.active[0]] + list(self.me.bench) if self.me.active else list(self.me.bench)
        board_op = [self.op.active[0]] + list(self.op.bench) if self.op.active else list(self.op.bench)
        for ai_idx, mp in enumerate(board_me):
            if mp is None: continue
            if ai_idx != 0 and not self.sw: break
            for aidx in range(2):
                r = self._base_atk(mp, aidx, ai_idx)
                if r is None: continue
                er, bd, bs = r
                ec = len(mp.energies)
                if aidx == 1 and ai_idx == 0 and ec >= 2 and not self.mb: break
                ne = False
                if ec < er:
                    if self.hc[C.BASIC_FIGHTING] >= 1 and not self.st.energyAttached:
                        ec += 1; ne = ec >= er
                    if not ne: continue
                for ti, op in enumerate(board_op):
                    if op is None: continue
                    if ti != 0 and not self.gust: break
                    dmg = bd
                    d = CARD_DB.get(op.id)
                    if d:
                        if d.weakness == EnergyType.FIGHTING: dmg *= 2
                        elif d.resistance == EnergyType.FIGHTING: dmg -= 30
                    sc = _tgt_score(op)
                    pr = _prizes(op) if op.hp <= dmg else 0
                    if pr == 0: sc *= dmg / max(1, op.hp)
                    if len(self.op.prize) <= pr: sc = 50000
                    sc += bs + (220 if ai_idx == 0 else 0) + (300 if ti == 0 else 0) + ec
                    if sc > best_s:
                        best_s = sc
                        best_p = Plan(ai_idx, ti, aidx, op.hp - dmg, ne)
        return best_p

    # ---------- energy targeting ----------
    def _e_sc(self, p, active):
        ec = len(p.energies); s = 8000 + (10 if active else 0)
        if p.id in (C.MAKUHITA, C.HARIYAMA):
            s += (1 if p.id == C.HARIYAMA else 0) + (100 if ec < 3 else 0) - (50 if self.rdy_har else 0)
        elif p.id == C.LUNATONE: s -= 100
        elif p.id == C.SOLROCK: s += 20 if ec < 1 else -100
        elif p.id in (C.RIOLU, C.MEGA_LUCARIO_EX):
            s += (1 if p.id == C.MEGA_LUCARIO_EX else 0) + (100 if ec < 2 else 0) - (50 if self.rdy_luc else 0)
        return s

    def _low(self): return self.me.deckCount <= LOW_DECK

    # ---------- option scoring ----------
    def _sc(self, o):
        t = o.type
        if t == OptionType.NUMBER: return o.number
        if t == OptionType.YES: return 100 if self.ctx == SelectContext.IS_FIRST else 1
        if t == OptionType.NO: return 0
        if t == OptionType.CARD:   return self._sc_card(o)
        if t == OptionType.PLAY:   return self._sc_play(o)
        if t == OptionType.ATTACH: return self._sc_attach(o)
        if t == OptionType.EVOLVE: return self._sc_evolve(o)
        if t == OptionType.ABILITY: return self._sc_ability(o)
        if t == OptionType.RETREAT:
            return 2000 if _plan and _plan.atk >= 1 else -1
        if t == OptionType.ATTACK:
            if _plan and _plan.aidx == 1: return 1100 if o.attackId == MEGA_BRAVE_ID else 1000
            return 1100 if o.attackId != MEGA_BRAVE_ID else 1000
        return 0

    def _sc_card(self, o):
        c = _get(self.obs, o.area, o.index, o.playerIndex)
        if c is None: return 0
        if self.ctx in (SelectContext.SWITCH, SelectContext.TO_ACTIVE):
            if o.playerIndex != self.mi:
                return 100 if _plan and o.index == _plan.tgt - 1 else 0
            if not isinstance(c, Pokemon): return 0
            s = len(c.energies) * 2
            if _plan and o.index == _plan.atk - 1: s += 100
            if c.id == C.MEGA_LUCARIO_EX: s += 8 if self.mp in (2,3) else 20
            elif c.id == C.HARIYAMA and len(c.energies) >= 2: s += 15
            elif c.id == C.MAKUHITA and len(c.energies) >= 2: s += 10
            elif c.id == C.SOLROCK: s += 5
            elif c.id == C.RIOLU: s += 4
            return s
        if self.ctx == SelectContext.SETUP_ACTIVE_POKEMON:
            if c.id == C.SOLROCK: return 2 if self.st.firstPlayer == self.mi else 4
            if c.id == C.RIOLU: return 3
            if c.id == C.MAKUHITA: return 1
            return 0
        if self.ctx == SelectContext.TO_HAND:
            s = 200 - self.hc.get(c.id, 0) * 100
            cid = c.id
            if cid == C.MAKUHITA: s += -10 if self.fc[cid] >= 1 else 10
            elif cid == C.HARIYAMA: s += 20 if self.fc[C.MAKUHITA] >= 1 else -20
            elif cid == C.LUNATONE: s += -250 if self.fc[cid] >= 1 else 60
            elif cid == C.SOLROCK: s += -250 if self.fc[cid] >= 1 else 50
            elif cid == C.RIOLU:
                ll = self.fc[C.RIOLU] + self.fc[C.MEGA_LUCARIO_EX]
                s += -150 if ll >= 2 else (-3 if ll >= 1 else 40)
            elif cid == C.MEGA_LUCARIO_EX: s += 40 if self.fc[C.RIOLU] >= 1 else -15
            elif cid == C.BASIC_FIGHTING: s += 30 if not _ab_used or not self.st.energyAttached else -1
            return s
        if self.ctx == SelectContext.ATTACH_FROM and isinstance(c, Pokemon):
            return self._e_sc(c, o.area == AreaType.ACTIVE)
        return 0

    def _sc_play(self, o):
        c = _get(self.obs, AreaType.HAND, o.index, self.mi)
        if c is None: return 0
        d = CARD_DB.get(c.id)
        if d and d.cardType == CardType.POKEMON:
            if c.id in (C.LUNATONE, C.SOLROCK) and self.fc[c.id] >= 1: return -1
            if c.id == C.RIOLU and self.fc[C.RIOLU] + self.fc[C.MEGA_LUCARIO_EX] >= 2: return -1
            return 20000
        # Trainer
        if c.id == C.SWITCH: return 6000 if _plan and _plan.atk > 0 else -1
        if c.id == C.PREMIUM_POWER_PRO:
            if self.st.supporterPlayed and _plan and _plan.rhp <= 0: return -1
            if not self.atk:
                ok = not self.st.supporterPlayed and self.hc[C.CARMINE] > 0 and self.hc[C.LILLIE_DETERMINATION] == 0 and not self._low()
                return 3050 if ok else -1
            return 5000
        if c.id == C.BOSS_ORDERS: return 3200 if _plan and _plan.tgt >= 1 else -1
        if c.id == C.CARMINE: return -1 if self._low() else 3000
        if c.id == C.LILLIE_DETERMINATION: return -1 if self._low() else 3100
        if c.id == C.GRAVITY_MOUNTAIN:
            has_s2 = any(p and CARD_DB.get(p.id) and CARD_DB[p.id].stage2
                         for p in (self.op.active + self.op.bench) if p)
            if has_s2: return 3500
            return 1200 if self.sid else -1
        return 10000

    def _sc_attach(self, o):
        c = _get(self.obs, AreaType.HAND, o.index, self.mi)
        p = _get(self.obs, o.inPlayArea, o.inPlayIndex, self.mi)
        if not isinstance(p, Pokemon) or c is None: return 0
        if c.id == C.HERO_CAPE:
            s = 7000
            if p.id == C.RIOLU: s += 100
            elif p.id == C.MEGA_LUCARIO_EX: s += 200
            return s
        s = self._e_sc(p, o.inPlayArea == AreaType.ACTIVE)
        bi = o.inPlayIndex if o.inPlayArea == AreaType.ACTIVE else o.inPlayIndex + 1
        if _plan and bi == _plan.atk and _plan.need_e: s += 200
        return s

    def _sc_evolve(self, o):
        p = _get(self.obs, o.inPlayArea, o.inPlayIndex, self.mi)
        if not isinstance(p, Pokemon): return 0
        if p.id == C.MAKUHITA and _plan and _plan.tgt == 0: return -1
        return 9000 + len(p.energies)

    def _sc_ability(self, o):
        c = _get(self.obs, o.area, o.index, self.mi)
        if c is None: return 0
        if c.id == C.LUMIOSE_CITY: return 1
        if c.id == C.LUNATONE and self._low(): return -1
        return 30000

    def _track_ab(self, ranked):
        global _ab_used
        if self.ctx != SelectContext.MAIN or not ranked: return
        o = self.sel.option[ranked[0]]
        if o.type == OptionType.ABILITY:
            c = _get(self.obs, o.area, o.index, self.mi)
            if c and c.id == C.LUNATONE: _ab_used = True

# ============================================================
# FORWARD SEARCH
# ============================================================
def _eval_state(obs):
    st = obs.current
    if st is None: return 0.0
    me = st.players[st.yourIndex]; op = st.players[1 - st.yourIndex]
    v  = (len(op.prize) - len(me.prize)) * 10000.0
    for p in ([me.active[0]] if me.active else []) + list(me.bench):
        if p is None: continue
        v += len(p.energies) * 120.0
        if p.id == C.MEGA_LUCARIO_EX: v += 400
        elif p.id == C.HARIYAMA: v += 200
    if me.active and me.active[0]: v += me.active[0].hp
    if op.active and op.active[0]: v -= op.active[0].hp * 1.5
    v += me.handCount * 5
    return v

def _search(obs_dict, obs):
    if not (_SEARCH_OK and USE_SEARCH): return None
    sel = obs.select
    if sel is None or sel.context != SelectContext.MAIN: return None
    sbi = getattr(obs, "search_begin_input", None) or obs_dict.get("search_begin_input")
    if sbi is None: return None
    base = Policy(obs).choose()
    cands = base[:SEARCH_CANDS]
    best_i, best_v = None, float("-inf")
    t0 = time.time()
    for first in cands:
        if time.time() - t0 > SEARCH_BUDGET: break
        sid = None
        try:
            res = search_begin(sbi)
            if getattr(res, "error", 0) != 0 or res.state is None: return None
            sid = res.state.searchId; cur = res.state.observation
            sel_a = [first]; steps = 0
            while steps < SEARCH_DEPTH:
                ar = search_step(sid, sel_a)
                if getattr(ar, "error", 0) != 0 or ar.state is None: break
                cur = ar.state.observation
                if cur.select is None or cur.current is None: break
                if cur.current.result is not None and cur.current.result != -1: break
                if cur.current.yourIndex != obs.current.yourIndex: break
                sub = Policy(cur).choose()
                if cur.select.context != SelectContext.MAIN:
                    sel_a = sub[:max(1, cur.select.minCount)]
                else:
                    sel_a = [sub[0]]
                    if cur.select.option[sub[0]].type == OptionType.END:
                        ar2 = search_step(sid, sel_a)
                        if ar2.state: cur = ar2.state.observation
                        break
                steps += 1
            v = _eval_state(cur)
            if v > best_v: best_v, best_i = v, first
        except Exception:
            return None
        finally:
            try:
                if sid is not None: search_end()
            except Exception: pass
    if best_i is None: return None
    rest = [i for i in base if i != best_i]
    return [best_i] + rest

# ============================================================
# AGENT ENTRY POINT
# ============================================================
def agent(obs_dict: dict) -> list[int]:
    global _plan, _turn, _ab_used
    try:
        obs = to_observation_class(obs_dict)
    except Exception:
        return MY_DECK if obs_dict.get("select") is None else [0]
    if obs.select is None:
        return MY_DECK
    if _turn != obs.current.turn:
        _turn = obs.current.turn; _ab_used = False; _plan = Plan()
    try:
        ordered = None
        if USE_SEARCH:
            ordered = _search(obs_dict, obs)
        if ordered is None:
            ordered = Policy(obs).choose()
        n = len(obs.select.option)
        ordered = [i for i in ordered if 0 <= i < n]
        if not ordered: return list(range(min(1, n)))
        k = min(obs.select.maxCount, n)
        k = max(k, min(max(1, obs.select.minCount), n))
        return ordered[:k]
    except Exception:
        n = len(obs.select.option)
        k = max(1, obs.select.minCount) if n else 0
        return list(range(min(k, n)))


## v3 meta patch: Crustle routing + safer option selection

After reviewing the strongest public notebooks and discussion themes, the most useful low-risk upgrade is not a new deck: it is making the 915+ Lucario search baseline more robust against the visible meta.

This patch keeps the same proven Mega Lucario ex list and forward-search hook, then adds three targeted changes:

- **Crustle-aware planning**: avoid zero-damage ex attacks into Crustle (`345`) and route energy/search pressure toward the non-ex Makuhita -> Hariyama line.
- **Safer `minCount`/`maxCount` handling**: do not fill optional selections with negative-score actions just because `maxCount` allows them.
- **Anti-stall guard**: if a turn has too many main actions, choose `END` when legal to protect the 10-minute match clock.

The code below patches the generated `main.py` in-place, then compiles it before the archive is built.

In [ ]:
from pathlib import Path

main_path = Path("main.py")
text = main_path.read_text()

if "CRUSTLE_AWARE" not in text:
    patches = [
        (
            "MEGA_BRAVE_ID   = 983        # attack ID for Mega Brave\n",
            "MEGA_BRAVE_ID   = 983        # attack ID for Mega Brave\n"
            "CRUSTLE_AWARE   = True       # route around Crustle's anti-ex wall\n"
            "MAX_TURN_ACTIONS = 45        # anti-stall guard\n",
        ),
        (
            "    MEGA_LUCARIO_EX      = 678\n"
            "    BASIC_FIGHTING       = 6\n",
            "    MEGA_LUCARIO_EX      = 678\n"
            "    CRUSTLE              = 345\n"
            "    BASIC_FIGHTING       = 6\n",
        ),
        (
            "        self.sid = self.st.stadium[0].id if self.st.stadium else 0\n"
            "        self._count(); self._scan()\n",
            "        self.sid = self.st.stadium[0].id if self.st.stadium else 0\n"
            "        self.facing_crustle = any(p is not None and p.id == C.CRUSTLE for p in (self.op.active + self.op.bench))\n"
            "        self._count(); self._scan()\n",
        ),
        (
            "        scores = [self._sc(o) for o in self.sel.option]\n"
            "        ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)\n",
            "        scores = [self._sc(o) for o in self.sel.option]\n"
            "        self._last_scores = scores\n"
            "        ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)\n",
        ),
        (
            "                    d = CARD_DB.get(op.id)\n"
            "                    if d:\n"
            "                        if d.weakness == EnergyType.FIGHTING: dmg *= 2\n"
            "                        elif d.resistance == EnergyType.FIGHTING: dmg -= 30\n"
            "                    sc = _tgt_score(op)\n"
            "                    pr = _prizes(op) if op.hp <= dmg else 0\n"
            "                    if pr == 0: sc *= dmg / max(1, op.hp)\n"
            "                    if len(self.op.prize) <= pr: sc = 50000\n"
            "                    sc += bs + (220 if ai_idx == 0 else 0) + (300 if ti == 0 else 0) + ec\n",
            "                    d = CARD_DB.get(op.id)\n"
            "                    if d:\n"
            "                        if d.weakness == EnergyType.FIGHTING: dmg *= 2\n"
            "                        elif d.resistance == EnergyType.FIGHTING: dmg -= 30\n"
            "                    my_data = CARD_DB.get(mp.id)\n"
            "                    crustle_immune = (\n"
            "                        CRUSTLE_AWARE\n"
            "                        and op.id == C.CRUSTLE\n"
            "                        and my_data is not None\n"
            "                        and (my_data.ex or my_data.megaEx)\n"
            "                    )\n"
            "                    if crustle_immune:\n"
            "                        dmg = 0\n"
            "                    sc = _tgt_score(op)\n"
            "                    pr = _prizes(op) if op.hp <= dmg else 0\n"
            "                    if pr == 0: sc *= dmg / max(1, op.hp)\n"
            "                    if len(self.op.prize) <= pr: sc = 50000\n"
            "                    if crustle_immune:\n"
            "                        sc = -10000\n"
            "                    sc += bs + (220 if ai_idx == 0 else 0) + (300 if ti == 0 else 0) + ec\n",
        ),
        (
            "        if p.id in (C.MAKUHITA, C.HARIYAMA):\n"
            "            s += (1 if p.id == C.HARIYAMA else 0) + (100 if ec < 3 else 0) - (50 if self.rdy_har else 0)\n"
            "        elif p.id == C.LUNATONE: s -= 100\n"
            "        elif p.id == C.SOLROCK: s += 20 if ec < 1 else -100\n"
            "        elif p.id in (C.RIOLU, C.MEGA_LUCARIO_EX):\n"
            "            s += (1 if p.id == C.MEGA_LUCARIO_EX else 0) + (100 if ec < 2 else 0) - (50 if self.rdy_luc else 0)\n"
            "        return s\n",
            "        if p.id in (C.MAKUHITA, C.HARIYAMA):\n"
            "            s += (1 if p.id == C.HARIYAMA else 0) + (100 if ec < 3 else 0) - (50 if self.rdy_har else 0)\n"
            "            if self.facing_crustle and ec < 3:\n"
            "                s += 900\n"
            "        elif p.id == C.LUNATONE: s -= 100\n"
            "        elif p.id == C.SOLROCK: s += 20 if ec < 1 else -100\n"
            "        elif p.id in (C.RIOLU, C.MEGA_LUCARIO_EX):\n"
            "            s += (1 if p.id == C.MEGA_LUCARIO_EX else 0) + (100 if ec < 2 else 0) - (50 if self.rdy_luc else 0)\n"
            "            if self.facing_crustle:\n"
            "                s -= 500\n"
            "        return s\n",
        ),
        (
            "            if cid == C.MAKUHITA: s += -10 if self.fc[cid] >= 1 else 10\n"
            "            elif cid == C.HARIYAMA: s += 20 if self.fc[C.MAKUHITA] >= 1 else -20\n",
            "            if cid == C.MAKUHITA:\n"
            "                s += -10 if self.fc[cid] >= 1 else 10\n"
            "                if self.facing_crustle:\n"
            "                    s += 200\n"
            "            elif cid == C.HARIYAMA:\n"
            "                s += 20 if self.fc[C.MAKUHITA] >= 1 else -20\n"
            "                if self.facing_crustle:\n"
            "                    s += 250\n",
        ),
        (
            "        if c.id == C.BOSS_ORDERS: return 3200 if _plan and _plan.tgt >= 1 else -1\n",
            "        if c.id == C.BOSS_ORDERS:\n"
            "            if _plan and _plan.tgt >= 1:\n"
            "                return 3600 if self.facing_crustle else 3200\n"
            "            return -1\n",
        ),
        (
            "            if has_s2: return 3500\n"
            "            return 1200 if self.sid else -1\n",
            "            if has_s2: return 3500\n"
            "            if self.facing_crustle and self.sid:\n"
            "                return 3400\n"
            "            return 1200 if self.sid else -1\n",
        ),
        (
            "        if p.id == C.MAKUHITA and _plan and _plan.tgt == 0: return -1\n"
            "        return 9000 + len(p.energies)\n",
            "        if p.id == C.MAKUHITA and _plan and _plan.tgt == 0 and not self.facing_crustle: return -1\n"
            "        if p.id == C.MAKUHITA and self.facing_crustle: return 9600 + len(p.energies)\n"
            "        return 9000 + len(p.energies)\n",
        ),
        (
            "# ============================================================\n"
            "# AGENT ENTRY POINT\n"
            "# ============================================================\n",
            "def _normalize_selection(ordered, scores, select):\n"
            "    n = len(select.option)\n"
            "    minc = max(0, min(select.minCount, n))\n"
            "    maxc = max(minc, min(select.maxCount, n))\n"
            "    out = []\n"
            "    seen = set()\n"
            "    for i in ordered or []:\n"
            "        if not isinstance(i, int) or i < 0 or i >= n or i in seen:\n"
            "            continue\n"
            "        score = scores[i] if scores is not None and i < len(scores) else 0\n"
            "        if score > 0 or len(out) < minc:\n"
            "            out.append(i); seen.add(i)\n"
            "        if len(out) >= maxc:\n"
            "            break\n"
            "    for i in range(n):\n"
            "        if len(out) >= minc:\n"
            "            break\n"
            "        if i not in seen:\n"
            "            out.append(i); seen.add(i)\n"
            "    return out\n"
            "\n"
            "# ============================================================\n"
            "# AGENT ENTRY POINT\n"
            "# ============================================================\n",
        ),
        (
            "    try:\n"
            "        ordered = None\n"
            "        if USE_SEARCH:\n"
            "            ordered = _search(obs_dict, obs)\n"
            "        if ordered is None:\n"
            "            ordered = Policy(obs).choose()\n"
            "        n = len(obs.select.option)\n"
            "        ordered = [i for i in ordered if 0 <= i < n]\n"
            "        if not ordered: return list(range(min(1, n)))\n"
            "        k = min(obs.select.maxCount, n)\n"
            "        k = max(k, min(max(1, obs.select.minCount), n))\n"
            "        return ordered[:k]\n"
            "    except Exception:\n"
            "        n = len(obs.select.option)\n"
            "        k = max(1, obs.select.minCount) if n else 0\n"
            "        return list(range(min(k, n)))\n",
            "    try:\n"
            "        if (obs.current is not None and obs.select.context == SelectContext.MAIN\n"
            "                and getattr(obs.current, 'turnActionCount', 0) >= MAX_TURN_ACTIONS):\n"
            "            for i, opt in enumerate(obs.select.option):\n"
            "                if opt.type == OptionType.END:\n"
            "                    return [i]\n"
            "        ordered = None\n"
            "        if USE_SEARCH:\n"
            "            ordered = _search(obs_dict, obs)\n"
            "        policy = Policy(obs)\n"
            "        if ordered is None:\n"
            "            ordered = policy.choose()\n"
            "        else:\n"
            "            policy.choose()\n"
            "        scores = getattr(policy, '_last_scores', None)\n"
            "        result = _normalize_selection(ordered, scores, obs.select)\n"
            "        if result:\n"
            "            return result\n"
            "        n = len(obs.select.option)\n"
            "        return list(range(min(max(0, obs.select.minCount), n)))\n"
            "    except Exception:\n"
            "        n = len(obs.select.option)\n"
            "        k = max(0, obs.select.minCount) if n else 0\n"
            "        return list(range(min(k, n)))\n",
        ),
    ]

    for old, new in patches:
        if old not in text:
            raise RuntimeError("v3 patch anchor not found:\n" + old[:240])
        text = text.replace(old, new, 1)

    main_path.write_text(text)
    print("v3 patch applied: Crustle-aware routing, safer optional selections, anti-stall END guard")
else:
    print("v3 patch already present; leaving main.py unchanged")

compile(main_path.read_text(), "main.py", "exec")
print("main.py compiles after v3 patch")

## June 22 meta read: what changed after v3

The public ladder moved again, so I re-checked recent public notebooks, author leaderboard snapshots, and my own latest submissions.

Key observations:

- **Nithin's 1084.5 compact Lucario** is a strong public reference for a small Lucario shell with Crustle and water/Snover target tuning. My reproduction of that public package scored better than the v3 search notebook snapshot, so I treat it as the current Lucario baseline to beat.
- **Roman V9/V10** remains a good explanation source for retuned anti-Crustle Lucario, but in my current public run V9 landed below the compact Lucario candidate. It is useful as a safety and validation reference, not necessarily the active best slot.
- **Ryota's Alakazam public agent** is important because it represents a different axis from Lucario mirrors. The author leaderboard snapshot was stronger than the public Lucario references, and local smoke tests were clean, so it is a useful second-slot candidate.
- **tomatomato's Starmie/Froslass note** does not include a ready submission, but the strategic lesson is valuable: prize-card tracking matters when forward search depends on hidden deck/prize assumptions. Wrong prize inference can make search choose impossible lines.
- **Dragapult ex** is a serious strategic direction because it attacks the meta from a different angle: spread damage, Budew item lock, and energy denial. The public notebook is reproducible and passed local smoke tests, but it needs a longer evaluation before replacing a known-scoring slot.

Operational lesson:

This competition scores only the **latest two submissions**, and ratings move for many hours as more games are played. A single new test submission can accidentally push out a better evaluated agent. When testing a new candidate, submit it together with the current keeper agent so the latest two slots stay intentional.

## June 23 evaluation log: what the settled scores told us

After waiting for the ratings to move, the latest two-slot experiment produced:

| Submission | Public package | Settled publicScore | Read |
|---|---|---:|---|
| `submission.tar.gz` | Nithin 1084.5 compact Lucario | **866.8** | Best recent Lucario shell; keep as slot #1 |
| `submission_dragapult.tar.gz` | SK Arin Dragapult spread | **762.9** | Interesting meta read, but too weak to keep scored |
| earlier Roman V9 | retuned anti-Crustle Lucario | **795.8** | Useful reference, below Nithin |
| earlier v3 search notebook | patched 915+ Lucario | **744.8** | Good teaching baseline, not the active best slot |

### Decision for the next dual submission

We chose **imitate a stronger public notebook** rather than keep iterating on Dragapult:

1. **Slot A — Nithin 1084.5 compact Lucario**  
   Still our best settled Lucario result in this cycle. Compact target tuning beats a heavier search notebook snapshot on public rating.

2. **Slot B — Roman V10**  
   Highly voted public notebook that claims `LB 950+`. It ships a data-tuned deck, crash-safe wrapper, and Crustle-aware routing with `EXTRA_CONTEXTS=False` after the author found mirror regressions. This is a cleaner public imitation target than Dragapult at 762.9.

### Why not keep Dragapult?

Dragapult is strategically attractive against wall / Mist meta, but our reproduced public package only reached **762.9** after settlement. That is below both Nithin and Roman V9, so it does not deserve a scored slot yet. We keep the build locally for future local A/B, not as an active leaderboard bet.

### Why not submit this notebook's patched v3 agent directly?

The v3 patch notebook is still valuable as a **builder and teaching artifact**, but its own settled score trail is weaker than Nithin 1084.5. Public notebooks should be honest: document the best known public packages, then keep improving the builder here.

### Competition operations reminder

- Only the **latest two** submissions are scored.
- Early values like `600.0` are not meaningful; wait hours to a day before judging.
- When testing a new candidate, submit it together with the keeper agent so you do not accidentally drop the better slot.

## Dual-slot plan submitted on June 23

The two active submissions after this update are intentionally different axes:

| Slot | Agent family | Why it is in the pair |
|---|---|---|
| A | Nithin 1084.5 compact Lucario | Best settled Lucario result we reproduced; strong into mirror / Crustle / water lines |
| B | Roman V10 retuned Lucario | Top public imitation target with validated deck retune and Crustle routing |

This is **not** a duplicate pair. Both are Lucario shells, but Roman V10 and Nithin 1084.5 are different public packages with different deck counts and policy details. The goal is to keep one known-good compact shell while testing whether Roman's retuned list climbs higher on the public ladder.

### What we are *not* submitting yet

- **Dragapult ex** — kept for local experiments only until it beats Nithin in longer A/B
- **Alakazam** — different axis, but our first public run was too noisy to trust immediately
- **Starmie / prize-tracking search** — strategic north star from tomatomato's note, but no ready public submission package yet

### Next improvement loop after these two settle

1. If Roman V10 > Nithin, keep Roman and replace the weaker slot with a non-Lucario axis (Alakazam or Starmie-style agent).
2. If Nithin stays ahead, port Roman's deck retune ideas into the v3 patch builder and re-test locally before another public submit.
3. Add conservative prize tracking from the Starmie note before turning search back on in this notebook.

In [ ]:
import glob
import os
import shutil
import tarfile
from pathlib import Path

CG_CANDIDATES = [
    '/kaggle/input/competitions/pokemon-tcg-ai-battle/sample_submission/cg',
    '/kaggle/input/**/sample_submission/cg',
    '/kaggle/input/**/cg-lib/cg',
    '/kaggle/input/**/cg',
    'cg',
]

cg_path = None
for pattern in CG_CANDIDATES:
    matches = [m for m in glob.glob(pattern, recursive=True) if os.path.isdir(m)]
    if matches:
        cg_path = matches[0]
        break
if cg_path is None:
    raise FileNotFoundError('Could not find cg/. Attach the competition data source.')

if Path('cg').resolve() != Path(cg_path).resolve():
    if Path('cg').exists():
        shutil.rmtree('cg')
    shutil.copytree(cg_path, 'cg')

with tarfile.open('submission.tar.gz', 'w:gz') as tar:
    tar.add('main.py', arcname='main.py')
    tar.add('deck.csv', arcname='deck.csv')
    tar.add('cg', arcname='cg')

print('cg source:', cg_path)
print('created:', Path('submission.tar.gz').resolve())
print('size:', Path('submission.tar.gz').stat().st_size)


In [ ]:
import tarfile
with tarfile.open('submission.tar.gz', 'r:gz') as tar:
    names = tar.getnames()
print('has main.py:', 'main.py' in names)
print('has deck.csv:', 'deck.csv' in names)
print('has cg:', any(n.startswith('cg/') for n in names))
assert 'main.py' in names and 'deck.csv' in names and any(n.startswith('cg/') for n in names)


## Smoke Test

A tiny runtime check catches import errors and illegal selections without making the public notebook slow.

In [ ]:
import importlib.util
import random
from cg.api import to_observation_class
from cg.game import battle_start, battle_select, battle_finish

spec = importlib.util.spec_from_file_location('our_agent', 'main.py')
our = importlib.util.module_from_spec(spec)
spec.loader.exec_module(our)
DECK = our.agent({'select': None})
print('deck:', len(DECK), 'USE_SEARCH:', getattr(our, 'USE_SEARCH', None), '_SEARCH_OK:', getattr(our, '_SEARCH_OK', None))

def legal_k(o):
    n = len(o.select.option)
    k = min(o.select.maxCount, n)
    if k < o.select.minCount:
        k = min(o.select.minCount, n)
    return max(1, k), n

def random_agent(obs):
    o = to_observation_class(obs)
    if o.select is None:
        return DECK
    k, n = legal_k(o)
    return random.sample(range(n), min(k, n))

def play(a0, a1, max_steps=2500):
    obs, start = battle_start(DECK, DECK)
    if obs is None:
        return None, f'start_failed:{getattr(start, "errorType", None)}'
    try:
        for _ in range(max_steps):
            oc = to_observation_class(obs)
            res = oc.current.result if oc.current is not None else -1
            if res is not None and res >= 0:
                return res, ''
            obs = battle_select((a0 if oc.current.yourIndex == 0 else a1)(obs))
        return None, 'max_steps'
    except Exception as e:
        return None, f'{type(e).__name__}: {e}'
    finally:
        try:
            battle_finish()
        except Exception:
            pass

for label, opp in [('mirror', our.agent), ('random', random_agent)]:
    errors = []
    results = []
    for _ in range(2):
        res, err = play(our.agent, opp)
        results.append(res)
        if err:
            errors.append(err)
    print(label, 'results=', results, 'errors=', errors)
    assert not errors
